# DID Analysis (Systematic): How ChatGPT Affected Crowdfunding Success

## Research Question
Did the launch of ChatGPT (Nov 30, 2022) change how text quality affects crowdfunding success?

## Method
- **Treatment**: ChatGPT Launch (November 30, 2022)
- **Outcome**: Log pledged amount (USD)
- **Sample**: Keep one currency (avoid fx noise)
    (df['pledged'] > 0) drop projects that never raised a dime
- **Comparison**: Kickstarter vs Indiegogo platforms


In [1]:
# Step 1: Load Libraries, Clean Data, and Create Features
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import re
import warnings
warnings.filterwarnings('ignore')

# Load data with AI detection scores
df = pd.read_pickle('final_with_deberta_ai_score_20251003_151656.pkl')
print(f"Loaded {len(df):,} projects")


# Impute missing goals and create success indicator
print("Attempting to impute missing 'goal' values for Indiegogo projects...")

# Define the condition for projects that need their goal imputed
impute_mask = (
    (df['platform'] == 'Indiegogo') &
    (df['goal'].isna()) &
    (df['pledged'].notna()) &
    (df['pledged'] > 0) &
    (df['funds_raised_percent'].notna()) &
    (df['funds_raised_percent'] > 0)
)

# Calculate the goal in the project's native currency
# goal = total_pledged / (percent_funded_as_decimal)
df.loc[impute_mask, 'goal'] = df.loc[impute_mask, 'pledged'] / (df.loc[impute_mask, 'funds_raised_percent'] / 100)

num_imputed = impute_mask.sum()
if num_imputed > 0:
    print(f"✅ Successfully imputed {num_imputed:,} missing 'goal' values.")
else:
    print("⚠️ No missing goals could be imputed with the available data.")

# --- Create a unified funding percentage and a robust success indicator ---

# First, create the unified 'funding_percent' column
df['funding_percent'] = np.nan
ks_mask = df['platform'] == 'Kickstarter'
ig_mask = df['platform'] == 'Indiegogo'
df.loc[ks_mask, 'funding_percent'] = df.loc[ks_mask, 'percent_funded']
df.loc[ig_mask, 'funding_percent'] = df.loc[ig_mask, 'funds_raised_percent']

# Now, create the robust success indicator
# A project is successful if its state is 'successful' OR it met its funding goal.
# This correctly handles Indiegogo's flexible funding campaigns.
df['success_indicator'] = (
    (df['funding_percent'] >= 100) | 
    (df['state'] == 'successful')
).astype(int)

print("✅ Created unified 'funding_percent' and robust 'success_indicator' columns.")

# --- Create preparation_time variable ---
print("Creating 'preparation_time' variable...")
# Convert timestamps to compatible datetime objects (UTC)
# launched_at is typically Unix timestamp (seconds), created_at is ISO string
launched_dt = pd.to_datetime(df['launched_at'], unit='s', utc=True, errors='coerce')
created_dt = pd.to_datetime(df['created_at'], utc=True, errors='coerce')

df['preparation_time'] = (launched_dt - created_dt).dt.days
print(f"✅ Created 'preparation_time' (Mean: {df['preparation_time'].mean():.1f} days)")

# Display a summary of the new success indicator
print("\n--- Success Indicator Breakdown ---")
print(pd.crosstab(df['platform'], df['success_indicator']))


Loaded 64,488 projects
Attempting to impute missing 'goal' values for Indiegogo projects...
✅ Successfully imputed 14,223 missing 'goal' values.
✅ Created unified 'funding_percent' and robust 'success_indicator' columns.
Creating 'preparation_time' variable...
✅ Created 'preparation_time' (Mean: 48.1 days)

--- Success Indicator Breakdown ---
success_indicator      0      1
platform                       
Indiegogo          17048    209
Kickstarter        13328  33903


In [2]:
# Step 2: Create key variables

# Fix dates
def clean_timezone(date_str):
    if pd.isna(date_str):
        return pd.NaT
    return re.sub(r'[+-]\d{2}:\d{2}$', '', str(date_str))

df['created_at_parsed'] = pd.to_datetime(df['created_at'].apply(clean_timezone), errors='coerce')

# Create outcome variable: log pledged amount
ks_mask = df['platform'] == 'Kickstarter'
ig_mask = df['platform'] == 'Indiegogo'

df['pledged_amount_usd'] = 0
df.loc[ks_mask, 'pledged_amount_usd'] = (df.loc[ks_mask, 'percent_funded'] / 100 * df.loc[ks_mask, 'goal']).fillna(0)
df.loc[ig_mask, 'pledged_amount_usd'] = (df.loc[ig_mask, 'funds_raised_percent'] / 100 * df.loc[ig_mask, 'goal']).fillna(0)
df['log_pledged_amount'] = np.log(df['pledged_amount_usd'].clip(lower=1))

print("✅ Created outcome variable: log_pledged_amount")


✅ Created outcome variable: log_pledged_amount


In [3]:
# Step 3: Create unified categories

def create_unified_categories(df):
    # Using your exact category specification
    
    # Kickstarter mapping
    ks_mapping = {
        # 1. Creative / Visual Arts
        'Illustration': 'Creative_Visual_Arts', 'Digital Art': 'Creative_Visual_Arts', 'Art': 'Creative_Visual_Arts',
        'Animation': 'Creative_Visual_Arts', 'Painting': 'Creative_Visual_Arts', 'Sculpture': 'Creative_Visual_Arts',
        'Photography': 'Creative_Visual_Arts', 'Mixed Media': 'Creative_Visual_Arts', 'Design': 'Creative_Visual_Arts',
        'Fine Art': 'Creative_Visual_Arts', 'Public Art': 'Creative_Visual_Arts',
        
        # 2. Publishing & Writing
        'Comic Books': 'Publishing_Writing', 'Graphic Novels': 'Publishing_Writing', 'Fiction': 'Publishing_Writing',
        'Nonfiction': 'Publishing_Writing', 'Anthologies': 'Publishing_Writing', 'Zines': 'Publishing_Writing',
        'Poetry': 'Publishing_Writing', 'Literature': 'Publishing_Writing', 'Academic': 'Publishing_Writing',
        'Journals': 'Publishing_Writing', 'Comics': 'Publishing_Writing',
        
        # 3. Technology / Hardware / Gadgets
        'Product Design': 'Technology_Hardware', 'Hardware': 'Technology_Hardware', 'Apps': 'Technology_Hardware',
        'Gadgets': 'Technology_Hardware', 'DIY Electronics': 'Technology_Hardware', 'Wearables': 'Technology_Hardware',
        'Gaming Hardware': 'Technology_Hardware', 'Robotics': 'Technology_Hardware', '3D Printing': 'Technology_Hardware',
        'Software': 'Technology_Hardware', 'Technology': 'Technology_Hardware',
        
        # 4. Games & Toys
        'Playing Cards': 'Games_Toys', 'Tabletop Games': 'Games_Toys', 'Video Games': 'Games_Toys',
        'Mobile Games': 'Games_Toys', 'Live Games': 'Games_Toys', 'Puzzles': 'Games_Toys',
        'Toys': 'Games_Toys', 'Games': 'Games_Toys',
        
        # 5. Film, Music & Performance
        'Shorts': 'Film_Music_Performance', 'Drama': 'Film_Music_Performance', 'Comedy': 'Film_Music_Performance',
        'Horror': 'Film_Music_Performance', 'Documentary': 'Film_Music_Performance', 'Music': 'Film_Music_Performance',
        'Rock': 'Film_Music_Performance', 'Hip-Hop': 'Film_Music_Performance', 'Pop': 'Film_Music_Performance',
        'Jazz': 'Film_Music_Performance', 'Classical Music': 'Film_Music_Performance', 'Country & Folk': 'Film_Music_Performance',
        'Electronic Music': 'Film_Music_Performance', 'Indie Rock': 'Film_Music_Performance', 'Theater': 'Film_Music_Performance',
        'Performance Art': 'Film_Music_Performance', 'Festivals': 'Film_Music_Performance', 'Film & Video': 'Film_Music_Performance',
        'Musical': 'Film_Music_Performance', 'Webseries': 'Film_Music_Performance', 'Television': 'Film_Music_Performance',
        'Radio & Podcasts': 'Film_Music_Performance', 'Audio': 'Film_Music_Performance', 'Narrative Film': 'Film_Music_Performance',
        'Dance': 'Film_Music_Performance', 'Sound': 'Film_Music_Performance', 'Blues': 'Film_Music_Performance',
        'World Music': 'Film_Music_Performance', 'Latin': 'Film_Music_Performance', 'Metal': 'Film_Music_Performance',
        'Punk': 'Film_Music_Performance', 'R&B': 'Film_Music_Performance',
        
        # 6. Food & Lifestyle
        'Restaurants': 'Food_Lifestyle', 'Drinks': 'Food_Lifestyle', 'Food Trucks': 'Food_Lifestyle',
        'Vegan': 'Food_Lifestyle', 'Cookbooks': 'Food_Lifestyle', 'Candles': 'Food_Lifestyle',
        'Fashion': 'Food_Lifestyle', 'Apparel': 'Food_Lifestyle', 'Accessories': 'Food_Lifestyle',
        'Jewelry': 'Food_Lifestyle', 'Footwear': 'Food_Lifestyle', 'Cosmetics': 'Food_Lifestyle',
        'Food': 'Food_Lifestyle',
        
        # 7. Cause / Community / Education
        'Community Gardens': 'Community_Education', 'Faith': 'Community_Education', 'Social Practice': 'Community_Education',
        'Events': 'Community_Education', 'Community': 'Community_Education'
    }
    
    # Indiegogo mapping
    ig_mapping = {
        # 1. Creative / Visual Arts
        'Art': 'Creative_Visual_Arts', 'Photography': 'Creative_Visual_Arts', 'Dance & Theater': 'Creative_Visual_Arts',
        'Web Series & TV Shows': 'Creative_Visual_Arts',
        
        # 2. Publishing & Writing
        'Comics': 'Publishing_Writing', 'Writing & Publishing': 'Publishing_Writing', 'Blogs/Podcasts/Vlogs': 'Publishing_Writing',
        
        # 3. Technology / Hardware / Gadgets
        'Productivity': 'Technology_Hardware', 'Phones & Accessories': 'Technology_Hardware', 'Energy & Green Tech': 'Technology_Hardware',
        'Smart Home': 'Technology_Hardware', 'Computers': 'Technology_Hardware', 'IoT': 'Technology_Hardware',
        'Security': 'Technology_Hardware', 'Drones': 'Technology_Hardware', 'VR/AR': 'Technology_Hardware',
        'Transportation': 'Technology_Hardware',
        
        # 4. Games & Toys
        'Tabletop Games': 'Games_Toys', 'Video Games': 'Games_Toys', 'Toys': 'Games_Toys',
        
        # 5. Film, Music & Performance
        'Film': 'Film_Music_Performance', 'Music': 'Film_Music_Performance', 'Audio': 'Film_Music_Performance',
        'Web Series & TV': 'Film_Music_Performance', 'Dance & Theater': 'Film_Music_Performance',
        
        # 6. Food & Lifestyle
        'Food & Beverages': 'Food_Lifestyle', 'Fashion & Wearables': 'Food_Lifestyle', 'Home': 'Food_Lifestyle',
        'Beauty': 'Food_Lifestyle', 'Health & Fitness': 'Food_Lifestyle', 'Wellness': 'Food_Lifestyle',
        
        # 7. Cause / Community / Education
        'Education': 'Community_Education', 'Human Rights': 'Community_Education', 'Local Businesses': 'Community_Education',
        'Environment': 'Community_Education', 'Social Innovations': 'Community_Education', 'Culture': 'Community_Education'
    }
    
    # Apply mappings
    df['category_unified'] = 'Other'
    ks_mask = df['platform'] == 'Kickstarter'
    ig_mask = df['platform'] == 'Indiegogo'
    
    df.loc[ks_mask, 'category_unified'] = df.loc[ks_mask, 'category_name'].map(ks_mapping).fillna('Other')
    df.loc[ig_mask, 'category_unified'] = df.loc[ig_mask, 'category_parent_name'].map(ig_mapping).fillna('Other')
    
    return df

df = create_unified_categories(df)
print("✅ Created unified categories")
print(df['category_unified'].value_counts())


✅ Created unified categories
category_unified
Film_Music_Performance    15206
Other                     11130
Publishing_Writing         9461
Technology_Hardware        7332
Food_Lifestyle             7036
Creative_Visual_Arts       6832
Games_Toys                 5788
Community_Education        1703
Name: count, dtype: int64


In [4]:
# Step 4: Add Month and Year Controls for Date Fixed Effects
# This controls for seasonal patterns and year-to-year trends

print("📅 Adding month and year controls...")

df_analysis_base = df.copy()

# Create month and year variables for date controls
df_analysis_base['month'] = df_analysis_base['created_at_parsed'].dt.month
df_analysis_base['year'] = df_analysis_base['created_at_parsed'].dt.year

print(f"   Date range: {df_analysis_base['created_at_parsed'].min()} to {df_analysis_base['created_at_parsed'].max()}")
print(f"✅ Date controls created successfully!")


📅 Adding month and year controls...
   Date range: 2020-10-31 23:59:59 to 2024-12-10 23:59:59
✅ Date controls created successfully!


In [5]:
# Step 5: Create Final Analysis Sample

df_analysis = df_analysis_base[
    (df_analysis_base['currency'] == 'USD') &          # keep one currency (avoid fx noise)
    (df_analysis_base['pledged'] > 0) &                # drop projects that never raised a dime
    (df_analysis_base['text_quality'].notna()) &  
    (df_analysis_base['platform'] == 'Kickstarter') &      # ensure quality metric available
    (df_analysis_base['created_at_parsed'].notna())    # valid launch date
].copy()

# Create treatment variables
gpt_date = pd.Timestamp('2022-11-30')
df_analysis['PostGPT'] = (df_analysis['created_at_parsed'] > gpt_date).astype(int)
df_analysis['indiegogo'] = (df_analysis['platform'] == 'Indiegogo').astype(int)

# Add controls
df_analysis['log_goal'] = np.log(df_analysis['goal'].clip(lower=1))

print(f"✅ Analysis sample: {len(df_analysis):,} projects")
print("\nBreakdown:")
breakdown = pd.crosstab(df_analysis['platform'], df_analysis['PostGPT'], margins=True)
breakdown.columns = ['Pre-ChatGPT', 'Post-ChatGPT', 'Total']
print(breakdown)


✅ Analysis sample: 23,798 projects

Breakdown:
             Pre-ChatGPT  Post-ChatGPT  Total
platform                                     
Kickstarter         7405         16393  23798
All                 7405         16393  23798


# --- Analysis Stream 1: Full Sample Analysis (Original) ---

The following models are run on the full `df_analysis` sample, including both successful and unsuccessful projects.


# --- Analysis Stream 2: Successful Projects Only ---

This second stream of analysis focuses on the user's primary objective: to understand the dynamics of projects that have already met their funding goal. All models in this section are run on a filtered dataframe (`df_successful`) that includes only projects where `success_indicator == 1`.


In [8]:
# Step 6: Create a Sample of ONLY Successful Projects

print("="*80)
print("Creating a new sample containing ONLY successful projects...")

# Use the robust 'success_indicator' to filter the dataframe
df_successful = df_analysis[df_analysis['success_indicator'] == 1].copy()

print(f"✅ Successful projects sample: {len(df_successful):,} projects")
print("\nBreakdown of SUCCESSFUL projects:")
successful_breakdown = pd.crosstab(df_successful['platform'], df_successful['PostGPT'], margins=True)
successful_breakdown.columns = ['Pre-ChatGPT', 'Post-ChatGPT', 'Total']
print(successful_breakdown)
print("="*80)

# Export the analysis sample for RStudio
# We filter for Kickstarter only to keep the sample clean for mediation
df_export = df_analysis[df_analysis['platform'] == 'Kickstarter'].copy()
df_export.to_csv('mediation_analysis_data.csv', index=False)

print("✅ Data exported to 'mediation_analysis_data.csv'")


Creating a new sample containing ONLY successful projects...
✅ Successful projects sample: 18,906 projects

Breakdown of SUCCESSFUL projects:
             Pre-ChatGPT  Post-ChatGPT  Total
platform                                     
Kickstarter         5578         13328  18906
All                 5578         13328  18906
✅ Data exported to 'mediation_analysis_data.csv'


The following cells replicate the systematic regression analysis on the `df_successful` sample.


In [9]:
### Model 1 (Successful Sample): Basic DID Model

formula1_succ = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT"
model1_succ = smf.ols(formula1_succ, data=df_successful).fit()
print(model1_succ.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     12.72
Date:                Thu, 13 Nov 2025   Prob (F-statistic):           2.65e-08
Time:                        16:09:37   Log-Likelihood:                -35489.
No. Observations:               18906   AIC:                         7.099e+04
Df Residuals:                   18902   BIC:                         7.102e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 10

In [10]:
### Model 2 (Successful Sample): Adding Essential Project Controls

formula2_succ = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count"
model2_succ = smf.ols(formula2_succ, data=df_successful).fit()
print(model2_succ.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.618
Model:                            OLS   Adj. R-squared:                  0.617
Method:                 Least Squares   F-statistic:                     6103.
Date:                Thu, 13 Nov 2025   Prob (F-statistic):               0.00
Time:                        16:09:40   Log-Likelihood:                -26423.
No. Observations:               18906   AIC:                         5.286e+04
Df Residuals:                   18900   BIC:                         5.290e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  3

In [11]:
### Model 3 (Successful Sample): Adding Category Fixed Effects

formula3_succ = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count + C(category_unified)"
model3_succ = smf.ols(formula3_succ, data=df_successful).fit()
print(model3_succ.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.681
Model:                            OLS   Adj. R-squared:                  0.680
Method:                 Least Squares   F-statistic:                     3354.
Date:                Thu, 13 Nov 2025   Prob (F-statistic):               0.00
Time:                        16:09:42   Log-Likelihood:                -24721.
No. Observations:               18906   AIC:                         4.947e+04
Df Residuals:                   18893   BIC:                         4.957e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------

In [12]:
### Model 4 (Successful Sample): Adding Time Fixed Effects

formula4_succ = "log_pledged_amount ~ indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count + C(category_unified) + C(year):C(month)"
model4_succ = smf.ols(formula4_succ, data=df_successful).fit()
print(model4_succ.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.681
Model:                            OLS   Adj. R-squared:                  0.681
Method:                 Least Squares   F-statistic:                     840.5
Date:                Thu, 13 Nov 2025   Prob (F-statistic):               0.00
Time:                        16:09:47   Log-Likelihood:                -24693.
No. Observations:               18906   AIC:                         4.948e+04
Df Residuals:                   18857   BIC:                         4.987e+04
Df Model:                          48                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------

In [4]:
### Model 5 (Successful Sample): The Full Model with AI Score Interactions

formula5_succ = "log_pledged_amount ~ text_quality*PostGPT + PostGPT + ai_score*PostGPT + log_goal + C(category_unified) + word_count + C(year):C(month)"
model5_succ = smf.ols(formula5_succ, data=df).fit()
print(model5_succ.summary())


NameError: name 'smf' is not defined

In [ ]:
# --- Model Comparison (Successful Sample) ---
from statsmodels.iolib.summary2 import summary_col

# Define the key variables of interest for the summary table
regressors_to_show = [
    'text_quality',
    'indiegogo',
    'PostGPT',
    'text_quality:indiegogo',
    'text_quality:PostGPT',
    'indiegogo:PostGPT',
    'ai_score',
    'ai_score:indiegogo',
    'ai_score:PostGPT'
]

# Generate the summary table for the successful sample
results_table_succ = summary_col(
    results=[model1_succ, model2_succ, model3_succ, model4_succ, model5_succ],
    model_names=['Model 1\n(Basic)', 'Model 2\n(+Controls)', 'Model 3\n(+Category)', 'Model 4\n(+Time)', 'Model 5\n(+AI Score)'],
    stars=True,
    float_format='%0.3f',
    regressor_order=regressors_to_show,
    drop_omitted=True
)

print(results_table_succ)


# --- Systematic Regression Analysis ---

The following cells build our difference-in-differences model step-by-step. Each model adds a new layer of controls to test the robustness of our findings.


In [10]:
### Model 1: Basic DID Model
# Purpose: Test the raw, core hypothesis without any confounding factors.
# This model shows the fundamental interaction effects and answers the basic question: Is there a difference-in-differences effect at all?

formula1 = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT"
model1 = smf.ols(formula1, data=df_analysis).fit()
print(model1.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.101
Model:                            OLS   Adj. R-squared:                  0.101
Method:                 Least Squares   F-statistic:                     653.4
Date:                Tue, 04 Nov 2025   Prob (F-statistic):               0.00
Time:                        14:04:13   Log-Likelihood:                -83602.
No. Observations:               34866   AIC:                         1.672e+05
Df Residuals:                   34859   BIC:                         1.673e+05
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  8

In [11]:
### Model 2: Adding Essential Project Controls
# Purpose: To see if the basic effect holds after controlling for the most obvious project-level characteristics.
# A project's funding goal and the length of its description are powerful predictors of success.

formula2 = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count"
model2 = smf.ols(formula2, data=df_analysis).fit()
print(model2.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.354
Method:                 Least Squares   F-statistic:                     2389.
Date:                Tue, 04 Nov 2025   Prob (F-statistic):               0.00
Time:                        14:17:08   Log-Likelihood:                -77833.
No. Observations:               34865   AIC:                         1.557e+05
Df Residuals:                   34856   BIC:                         1.558e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  2

In [12]:
### Model 3: Adding Category Fixed Effects
# Purpose: To account for inherent differences between project types.
# This control ensures we are comparing like with like and that the observed effects aren't just due to shifts in project categories over time.

formula3 = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count + C(category_unified)"
model3 = smf.ols(formula3, data=df_analysis).fit()
print(model3.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.379
Model:                            OLS   Adj. R-squared:                  0.379
Method:                 Least Squares   F-statistic:                     1418.
Date:                Tue, 04 Nov 2025   Prob (F-statistic):               0.00
Time:                        14:19:22   Log-Likelihood:                -77148.
No. Observations:               34865   AIC:                         1.543e+05
Df Residuals:                   34849   BIC:                         1.545e+05
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------

In [13]:
### Model 4: Adding Time Fixed Effects (The Full DID Model for Text Quality)
# Purpose: To control for seasonality and underlying time trends unrelated to ChatGPT.
# This is the most robust version of the primary DID model. It isolates the PostGPT treatment effect from any general trends.

formula4 = "log_pledged_amount ~ text_quality*indiegogo + text_quality*PostGPT + indiegogo*PostGPT + log_goal + word_count + C(category_unified) + C(year):C(month)"
model4 = smf.ols(formula4, data=df_analysis).fit()
print(model4.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.381
Model:                            OLS   Adj. R-squared:                  0.380
Method:                 Least Squares   F-statistic:                     329.5
Date:                Tue, 04 Nov 2025   Prob (F-statistic):               0.00
Time:                        14:19:39   Log-Likelihood:                -77093.
No. Observations:               34865   AIC:                         1.543e+05
Df Residuals:                   34799   BIC:                         1.549e+05
Df Model:                          65                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------

In [20]:
### Model 5: The Full Model with AI Score Interactions
# Purpose: To test whether the effects we attributed to text_quality are actually better explained by ai_score.
# This model directly pits the "text quality" hypothesis against the "AI-likeness" hypothesis.

formula2 = """
success_indicator ~ 
text_quality + indiegogo + PostGPT +
indiegogo:text_quality + 
indiegogo:PostGPT + indiegogo:PostGPT:text_quality +
text_quality:PostGPT + 
ai_score + ai_score:PostGPT +
log_goal + C(category_unified) + word_count + C(year):C(month)
"""
model5 = smf.ols(formula5, data=df_analysis).fit()
print(model5.summary())


                            OLS Regression Results                            
Dep. Variable:     log_pledged_amount   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     319.9
Date:                Tue, 04 Nov 2025   Prob (F-statistic):               0.00
Time:                        16:13:54   Log-Likelihood:                -77188.
No. Observations:               34865   AIC:                         1.545e+05
Df Residuals:                   34798   BIC:                         1.551e+05
Df Model:                          66                                         
Covariance Type:            nonrobust                                         
                                                    coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------

In [16]:
# --- Model Comparison ---
from statsmodels.iolib.summary2 import summary_col

# Define the key variables of interest for the summary table
# We focus on the main effects and the highest-order interaction terms
regressors_to_show = [
    'text_quality',
    'indiegogo',
    'PostGPT',
    'text_quality:indiegogo',
    'text_quality:PostGPT',
    'indiegogo:PostGPT',
    'ai_score',
    'ai_score:indiegogo',
    'ai_score:PostGPT'
]

# Generate the summary table
# This table shows how coefficients and significance change as we add more controls
results_table = summary_col(
    results=[model1, model2, model3, model4, model5],
    model_names=['Model 1\n(Basic)', 'Model 2\n(+Controls)', 'Model 3\n(+Category)', 'Model 4\n(+Time)', 'Model 5\n(+AI Score)'],
    stars=True,
    float_format='%0.3f',
    regressor_order=regressors_to_show,
    drop_omitted=True # Hide variables not in our list
)

print(results_table)



                        Model 1    Model 2     Model 3    Model 4    Model 5  
                        (Basic)  (+Controls) (+Category)  (+Time)  (+AI Score)
------------------------------------------------------------------------------
text_quality           -0.371    0.069       0.697       0.667     0.825      
                       (0.824)   (0.699)     (0.686)     (0.686)   (0.686)    
indiegogo              1.570*    -0.949      -1.037      -1.117    -1.062     
                       (0.905)   (0.768)     (0.753)     (0.754)   (0.753)    
PostGPT                4.052***  3.959***    3.659***    4.861***  4.807***   
                       (0.891)   (0.756)     (0.741)     (0.895)   (0.894)    
text_quality:indiegogo -0.900    -1.176      -1.093      -1.022    -1.213     
                       (0.931)   (0.789)     (0.774)     (0.774)   (0.776)    
text_quality:PostGPT   -3.825*** -3.675***   -3.497***   -3.508*** -3.177***  
                       (0.918)   (0.778)     (0.763

In [ ]:
# Save final cleaned data from the analysis
df_analysis.to_csv('did_analysis_final_clean_systematic.csv', index=False)
df_analysis.to_pickle('did_analysis_final_clean_systematic.pkl')

print("✅ Final cleaned data saved to:")
print("   - did_analysis_final_clean_systematic.csv")
print("   - did_analysis_final_clean_systematic.pkl")
